# FedMed-FL on Google Colab — REAL Kaggle Data

Federated learning across 3 simulated hospitals, trained on a real public Kaggle symptom dataset.

**Runtime -> Change runtime type -> CPU is fine.** The model is small; a GPU changes nothing.

---

## Read this first — it will save your results section

The most popular Kaggle disease dataset (`kaushil268/disease-prediction-using-machine-learning`) is a **deterministic rule table, not clinical data**. Measured on the actual file:

| property | value |
|---|---|
| rows | 4,920 |
| **unique symptom patterns** | **304** |
| patterns mapping to >1 disease | **0** |
| plain logistic-regression accuracy | **1.0000** |

Train on it raw and your federated, centralized, isolated, DP and no-DP runs will **all** score ~1.00. The experiment would distinguish nothing, and a perfect confusion-matrix diagonal is the first thing an examiner questions.

This notebook handles it two ways:

1. **Default to `large`** = `dhivyeshrk/diseases-and-symptoms-dataset` (~246k rows, 773 diseases) — big and messy enough to be a real benchmark.
2. **`inject_ehr_realism()`** — clinically-motivated corruption (symptom recall bias, charting false positives, provisional-diagnosis label noise) that breaks the lookup-table property.

Cell 4 audits whatever dataset you load and shouts if it is degenerate.

## 1. Get the project files

Pick **one** option.

In [ ]:
# OPTION A — upload FedMed-FL.zip (simplest; lost when the runtime restarts)
from google.colab import files
up = files.upload()               # choose FedMed-FL.zip
!unzip -q -o FedMed-FL.zip -d /content
PROJECT = '/content/FedMed-FL'


In [ ]:
# OPTION B — keep it in Google Drive (persists across sessions)
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = '/content/drive/MyDrive/FedMed-FL'   # <-- edit this path


In [ ]:
import os, sys
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
print('working in:', os.getcwd())
print(sorted(os.listdir('.')))


In [ ]:
# Colab already ships torch / numpy / pandas / sklearn / matplotlib.
!pip install -q kagglehub 'flwr>=1.16,<2.0'
import torch; print('torch', torch.__version__)


## 2. Kaggle credentials

Get a token: <https://www.kaggle.com/settings> -> **API** -> *Create New Token*. That downloads `kaggle.json`.

**Best option — Colab Secrets** (survives restarts, never appears in your notebook):
click the **key icon** in the left sidebar, add two secrets, and toggle notebook access on:

| Name | Value |
|---|---|
| `KAGGLE_USERNAME` | the `username` field from kaggle.json |
| `KAGGLE_KEY` | the `key` field from kaggle.json |

Then just run the next cell. If you would rather upload the file, use the cell after it.

In [ ]:
from src.kaggle_data import ensure_kaggle_auth
ensure_kaggle_auth()


In [ ]:
# Fallback — upload kaggle.json directly
# from google.colab import files
# files.upload()                # choose kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# from src.kaggle_data import ensure_kaggle_auth; ensure_kaggle_auth()


## 3. Download and inspect the dataset

In [ ]:
from src import kaggle_data

print('verified dataset slugs available:')
for k, v in kaggle_data.DATASETS.items():
    print(f'  {k:<9} {v}')


In [ ]:
WHICH = 'large'        # 'large' (recommended) | 'classic' | 'itachi' | 'profile'
TOP_K  = 12            # keep the K most frequent diseases
ALPHA  = 0.4           # Dirichlet non-IID skew: 0.1 extreme, 100 = IID

path = kaggle_data.download(kaggle_data.DATASETS[WHICH])
csv  = kaggle_data.find_main_csv(path)
print('using CSV:', csv)


## 4. Audit — is this dataset actually learnable?

Never skip this. If it prints the degeneracy warning, your numbers will be meaningless.

In [ ]:
X_df, y_s, sym_cols = kaggle_data.load_symptom_csv(csv, top_k_classes=TOP_K)
audit = kaggle_data.audit_dataset(X_df, y_s)


## 5. Build the 3-hospital federated split

A Kaggle table has no geography, so the non-IID skew comes from a Dirichlet partition instead of from climate zones as in the synthetic track.

**State this in your report:** a Kaggle symptom table has no admission dates and no clinician confidence scores, so the Phase-1 quarantine gate and the Phase-2 schema / physiological-bounds checks are **skipped** here — only the loss-based label filter runs. Fabricating admission metadata to make those stages look busy would be inventing clinical data.

In [ ]:
bundle = kaggle_data.build_kaggle_clients(
    WHICH, csv_path=csv, top_k_classes=TOP_K, alpha=ALPHA,
    realism=True,        # clinically-motivated corruption, see docstring
    max_rows=60000)      # laptop-scale; raise if you have time

import pandas as pd
pd.DataFrame({cid: pd.Series(c['y']).value_counts() for cid, c in bundle['clients'].items()}).fillna(0).astype(int).T


## 6. Main experiment — federated + record-level DP

This is the number you report.

In [ ]:
# Shell magics do not honour backslash continuation — keep each command on ONE line.
!python src/run_simulation.py --data kaggle --kaggle-which {WHICH} --top-k-classes {TOP_K} --alpha {ALPHA} --rounds 20


## 7. The comparisons that make it a paper

Federated vs isolated vs centralized, and the cost of privacy.

In [ ]:
# Utility ceiling — federation without differential privacy
!python src/run_simulation.py --data kaggle --kaggle-which {WHICH} --top-k-classes {TOP_K} --alpha {ALPHA} --rounds 20 --no-dp


In [ ]:
# Centralized pooling — what you could do if HIPAA/GDPR/DISHA did not exist
!python src/run_centralized.py --data kaggle --kaggle-which {WHICH} --top-k-classes {TOP_K} --alpha {ALPHA}


In [ ]:
# How does heterogeneity hurt? Sweep the Dirichlet alpha.
# subprocess is used instead of ! so the loop variable substitutes reliably.
import subprocess, sys

for a in [0.1, 0.5, 1.0, 100.0]:
    print(f'\n===================== alpha = {a} =====================')
    out = subprocess.run(
        [sys.executable, 'src/run_simulation.py', '--data', 'kaggle',
         '--kaggle-which', WHICH, '--top-k-classes', str(TOP_K),
         '--alpha', str(a), '--rounds', '15', '--no-dp'],
        capture_output=True, text=True)
    for line in out.stdout.splitlines():
        if 'FINAL' in line or 'isolated' in line or 'Federated global' in line:
            print(' ', line.strip())


In [ ]:
# Privacy-utility trade-off curve
!python src/run_simulation.py --data kaggle --kaggle-which {WHICH} --top-k-classes {TOP_K} --alpha {ALPHA} --privacy-sweep


## 8. If DP collapses, this is why

DP-SGD needs **thousands of records per hospital**. Measured on the small `classic` dataset (~1,000 cases/hospital):

| setting | accuracy |
|---|---|
| no DP, 12 classes | 0.766 |
| DP sigma=1.5, 12 classes | 0.270 |
| no DP, 41 classes | 0.637 |
| DP sigma=1.5, 41 classes | 0.087 |

The pipeline is fine — there is simply not enough data to average the noise away. Fixes, in order of preference: use `WHICH='large'`; raise `max_rows`; reduce `TOP_K`; or run `--no-dp` and say so explicitly in your report.

## 9. The synthetic track — do not drop it

**No public Kaggle symptom dataset has region or season metadata.** Your proposal's context-aware inference claim (Dengue in monsoon Delhi vs a sub-zero region) therefore *cannot* be validated on Kaggle data.

Run both tracks and report both. That is a stronger paper than either alone:

- **Kaggle track** — proves the FL + DP + filtering machinery works on real third-party data
- **Synthetic track** — proves the context-aware contribution, which needs geography no public dataset provides

In [ ]:
# Synthetic track: context-aware, 8 diseases, region x season priors
!python src/run_simulation.py --rounds 20
!python src/cdss.py


## 10. Collect the figures and download everything

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('outputs/*.png')):
    print(f); display(Image(f, width=760))


In [ ]:
!zip -qr fedmed_outputs.zip outputs
from google.colab import files
files.download('fedmed_outputs.zip')


## 11. What Colab cannot do

The real two-laptop Flower deployment needs two machines on the same network; Colab cannot open a gRPC socket between them. Run `federated/server.py` and `federated/client.py` locally in VS Code — README Section 6.

You can still exercise the Flower client logic here without the network:

In [ ]:
import numpy as np, config as cfg
from federated.client import HospitalClient
from federated.server import build_test_loader
from src.model import build_model, set_weights
from src import engine, data_utils, aggregate

test_loader = build_test_loader()
gm = build_model(data_utils.NUM_FEATURES, cfg.SEED)
params = [v.cpu().numpy() for v in gm.state_dict().values()]
clients = [HospitalClient(c) for c in cfg.CLIENT_IDS]

for rnd in range(1, 6):
    conf = {'server_round': rnd, 'local_epochs': 3, 'dp_mode': 'record',
            'sigma': 1.5, 'clip_norm': 1.0}
    ws, ns = [], []
    for c in clients:
        w, n, _ = c.fit(params, conf)
        ws.append([np.asarray(x) for x in w]); ns.append(n)
    params = aggregate.fedavg(ws, ns)
    set_weights(gm, params)
    ev = engine.evaluate(gm, test_loader)
    print(f"ROUND {rnd}: acc={ev['accuracy']:.4f}  macro-F1={ev['macro_f1']:.4f}")
